# Kaggle Evaluation: Stage 2 SemEval 2014

**Metrics:** ACSA Gold-Category Sentiment Accuracy
**Checkpoints:** Uploaded to dataset `stage2-sem14-eval-checkpoints`

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml
import os, sys, json, shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
# --- Wire SemEval 2014 XMLs ---
KAGGLE_INPUT = None
for candidate in ['/kaggle/input/semeval-2014-absa-restaurant',
                  '/kaggle/input/datasets/lcminhc/semeval-2014-absa-restaurant',
                  '/kaggle/input/datasets/duclm318/semeval-2014-absa-restaurant']:
    if os.path.exists(candidate):
        KAGGLE_INPUT = candidate
        break
assert KAGGLE_INPUT, 'Dataset semeval-2014-absa-restaurant not found'

os.makedirs('SemEval-2014', exist_ok=True)
shutil.copy(f'{KAGGLE_INPUT}/Restaurants_Train.xml', 'SemEval-2014/Restaurants_Train.xml')
shutil.copy(f'{KAGGLE_INPUT}/Restaurants_Test_Gold.xml', 'SemEval-2014/Restaurants_Test_Gold.xml')

# --- Prepare data ---
!python scripts/01_prepare_data.py

# --- Wire p5-embed-v4 ---
EMB = None
for candidate in ['/kaggle/input/p5-embed-v4',
                  '/kaggle/input/datasets/lcminhc/p5-embed-v4',
                  '/kaggle/input/datasets/duclm318/p5-embed-v4']:
    if os.path.exists(candidate):
        EMB = candidate
        break
assert EMB, 'Dataset p5-embed-v4 not found'

os.makedirs('checkpoints/embedding_2014', exist_ok=True)
shutil.copy(f'{EMB}/embedding_v4_s2_best.pt', 'checkpoints/embedding_2014/best.pt')

# --- Wire Checkpoints ---
CKPTS = None
for candidate in ['/kaggle/input/stage2-sem14-eval-checkpoints',
                  '/kaggle/input/datasets/duclm318/stage2-sem14-eval-checkpoints']:
    if os.path.exists(candidate):
        CKPTS = candidate
        break
assert CKPTS, 'Dataset stage2-sem14-eval-checkpoints not found'

os.makedirs('checkpoints/stage2_2014_noret', exist_ok=True)
os.makedirs('checkpoints/stage2_2014_auxloss', exist_ok=True)
shutil.copy(f'{CKPTS}/stage2_2014_noret.pt', 'checkpoints/stage2_2014_noret/best.pt')
shutil.copy(f'{CKPTS}/stage2_2014_auxloss.pt', 'checkpoints/stage2_2014_auxloss/best.pt')

## 1. Build FAISS Index

In [ ]:
os.makedirs('indexes', exist_ok=True)

!python scripts/03_build_index.py \
    --embedding_ckpt checkpoints/embedding_2014/best.pt \
    --input data/processed/sentiment_records.jsonl \
    --out_dir indexes/

## 2. Evaluate Gold-Category Accuracy

In [ ]:
import subprocess

EVAL_EXPERIMENTS = [
    {"name": "No-Retrieval", "ckpt_dir": "checkpoints/stage2_2014_noret", "config": "configs/stage2_2014_noret.yaml", "no_retrieval": True},
    {"name": "Aux Loss", "ckpt_dir": "checkpoints/stage2_2014_auxloss", "config": "configs/stage2_2014_auxloss.yaml", "no_retrieval": False},
]

print("=" * 60)
print("GOLD-CATEGORY ACCURACY (SemEval 2014 Test Set)")
print("=" * 60)

for exp in EVAL_EXPERIMENTS:
    if not os.path.exists(f'{exp["ckpt_dir"]}/best.pt'):
        print(f'SKIP {exp["name"]} — checkpoint missing'); continue
        
    cmd = ["python", "scripts/06_evaluate_sentiment_only.py",
           "--stage2_ckpt", f'{exp["ckpt_dir"]}/best.pt',
           "--stage2_config", exp["config"],
           "--data_dir", "data/processed"]
    if exp["no_retrieval"]:
        cmd.append("--no_retrieval")
    else:
        cmd += ["--embedding_ckpt", "checkpoints/embedding_2014/best.pt", "--index_dir", "indexes/"]
    
    print(f'\n--- {exp["name"]} ---')
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("ERROR:")
        print(result.stderr)
    else:
        print(result.stdout)